# Best Student Architecture per Teacher

**MO434 deliverable.**  
This is a self-contained notebook that builds and trains the *best student* found in the study, **from scratch**, for a given teacher.  

Across both datasets, the strongest distilled student uses:
- **pre-GAP feature-map target** with a **convolutional predictor**
- **MSE + CrossEntropy** objective
- the largest encoder **Student-L** (`Conv32 → Conv64 → Conv128 → Conv256 → Conv512`). 

The teacher only changes *which* representation is imitated, so the same student is trained once per teacher.

The notebook runs three phases end to end:

1. **Teacher** — adapt an ImageNet-pretrained backbone to the dataset (freeze encoder, train the classifier head, then briefly fine-tune the last block).
2. **Student** — define the Student-L encoder + convolutional predictor and distill it against the frozen teacher with `MSE + CE`.
3. **Evaluation** — test Top-1/Top-5 plus parameter and GFLOPs savings versus the teacher.  

**Inputs:** none besides the raw dataset (auto-downloaded via torchvision) and the `TEACHER` /`DATASET` choice in the config cell. 

**Outputs:** written to `notebooks/outputs/` — the trained student's weights (`{dataset}_{teacher}_student_l_pregap_mse_ce.pt`) and a metrics summary (`..._summary.json`) with test Top-1/Top-5 and parameter/GFLOPs savings vs. the teacher.

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from torch import Tensor, nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import (
    ConvNeXt_Tiny_Weights, ResNet50_Weights, convnext_tiny, resnet50,
)
from tqdm.auto import tqdm

# ── Choose the experiment ─────────────────────────────────────────────────────
DATASET = "aircraft"        # "aircraft" or "food101"
TEACHER = "convnext_tiny"   # "convnext_tiny" or "resnet50"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Auto-detect data root ─────────────────────────────────────────────────────
DATA_ROOT = Path("../data")

if not DATA_ROOT.exists():
    if Path("/content").exists():  # Google Colab
        DATA_ROOT = Path("/content/data")
    else:                          # regular standalone Jupyter notebook
        DATA_ROOT = Path("./data")

DATA_ROOT.mkdir(parents=True, exist_ok=True)
print(f"data_root={DATA_ROOT.resolve()}")

# ── Best-student recipe (fixed by the study) ──────────────────────────────────
STUDENT_CHANNELS = [32, 64, 128, 256, 512]  # Student-L encoder
TARGET = "pregap"                            # convolutional predictor on the feature map
ALPHA, BETA = 1.0, 1.0                       # MSE + CrossEntropy weights

# ── Training hyper-parameters ─────────────────────────────────────────────────
IMAGE_SIZE, BATCH_SIZE, NUM_WORKERS = 224, 32, 4
TEACHER_STAGE1_EPOCHS, TEACHER_STAGE2_EPOCHS, TEACHER_STAGE2_LR = 30, 15, 5e-5
DISTILL_EPOCHS, LR, WEIGHT_DECAY = 30, 1e-3, 1e-4
SEED = 42
TRAIN_LIMIT = None  # set to a small int (e.g. 200) for a quick smoke test

NUM_CLASSES = {"aircraft": 100, "food101": 101}[DATASET]
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
print(f"dataset={DATASET} teacher={TEACHER} device={DEVICE}")

## 1. Data

FGVC-Aircraft uses its official splits (variant labels).  
Food-101 keeps the official test split and carves a validation set out of the training split (every tenth sample).  

Validation/test use a plain resize + normalize; training adds light augmentation.

In [ ]:
def make_transform(train: bool) -> transforms.Compose:
    ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE),
                             interpolation=transforms.InterpolationMode.BICUBIC)]
    if train:
        ops += [transforms.RandomHorizontalFlip(), transforms.RandAugment(num_ops=2, magnitude=9)]
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)


def limit(dataset, n):
    return dataset if n is None else Subset(dataset, range(min(n, len(dataset))))


def build_split(split: str):
    """Return the requested split ('train' | 'val' | 'test') for the chosen dataset."""
    train_tf = make_transform(train=split == "train")
    if DATASET == "aircraft":
        ds = datasets.FGVCAircraft(DATA_ROOT, split=split, annotation_level="variant",
                                   transform=train_tf, download=True)
        return limit(ds, TRAIN_LIMIT if split == "train" else None)
    # food101 has no official val split: hold out every tenth training sample.
    tv_split = "train" if split in {"train", "val"} else "test"
    ds = datasets.Food101(DATA_ROOT, split=tv_split, transform=train_tf, download=True)
    if split == "val":
        return Subset(ds, range(0, len(ds), 10))
    if split == "train":
        return limit(Subset(ds, [i for i in range(len(ds)) if i % 10 != 0]), TRAIN_LIMIT)
    return ds


loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
train_loader = DataLoader(build_split("train"), shuffle=True, **loader_kwargs)
val_loader = DataLoader(build_split("val"), shuffle=False, **loader_kwargs)
test_loader = DataLoader(build_split("test"), shuffle=False, **loader_kwargs)
print(f"train={len(train_loader.dataset)} val={len(val_loader.dataset)} test={len(test_loader.dataset)}")

## 2. Teacher

Both teachers expose the same interface: an `encoder` producing a spatial feature map, a pooling layer, and a linear `classifier`. 

The student later reuses the frozen `pool` and `classifier` as its head, so we keep them as separate attributes.

In [ ]:
class TeacherClassifier(nn.Module):
    def __init__(self, name: str, num_classes: int) -> None:
        super().__init__()
        if name == "resnet50":
            model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
            self.encoder = nn.Sequential(*list(model.children())[:-2])
            self.pool = model.avgpool
            feature_dim = model.fc.in_features
        elif name == "convnext_tiny":
            model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            self.encoder = model.features
            self.pool = nn.AdaptiveAvgPool2d(1)
            feature_dim = model.classifier[2].in_features
        else:
            raise ValueError(name)
        self.feature_channels = feature_dim  # channels of the pre-GAP feature map
        self.classifier = nn.Linear(feature_dim, num_classes)

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_last_block(self):
        for p in self.encoder[-1].parameters():
            p.requires_grad = True

    def forward_features(self, x: Tensor):
        feature_map = self.encoder(x)
        pooled = torch.flatten(self.pool(feature_map), 1)
        return feature_map, pooled

    def classify_feature_map(self, feature_map: Tensor) -> Tensor:
        return self.classifier(torch.flatten(self.pool(feature_map), 1))

    def forward(self, x: Tensor) -> Tensor:
        _, pooled = self.forward_features(x)
        return self.classifier(pooled)

In [ ]:
@torch.no_grad()
def accuracy(logits: Tensor, labels: Tensor):
    top5 = logits.topk(5, dim=1).indices
    correct = top5 == labels.unsqueeze(1)
    top1 = correct[:, 0].float().mean().item() * 100
    top5 = correct.any(dim=1).float().mean().item() * 100
    return top1, top5


def run_classifier_epoch(model, loader, optimizer, train: bool, desc: str):
    model.train(train)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    totals = {"loss": 0.0, "top1": 0.0, "n": 0}
    progress = tqdm(loader, desc=desc, leave=False, unit="batch")
    for images, labels in progress:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.set_grad_enabled(train), torch.amp.autocast(DEVICE.type, enabled=AMP):
            logits = model(images)
            loss = F.cross_entropy(logits, labels)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        bs = labels.size(0)
        totals["loss"] += loss.item() * bs
        totals["top1"] += accuracy(logits, labels)[0] * bs
        totals["n"] += bs
        progress.set_postfix(loss=f"{totals['loss'] / totals['n']:.3f}",
                             top1=f"{totals['top1'] / totals['n']:.2f}")
    return totals["loss"] / totals["n"], totals["top1"] / totals["n"]


def adapt_teacher() -> TeacherClassifier:
    """Two-stage adaptation: train the head, then fine-tune the last encoder block."""
    teacher = TeacherClassifier(TEACHER, NUM_CLASSES).to(DEVICE)
    best_state, best_top1 = None, -1.0

    def maybe_save(top1):
        nonlocal best_state, best_top1
        if top1 > best_top1:
            best_top1 = top1
            best_state = {k: v.detach().cpu().clone() for k, v in teacher.state_dict().items()}

    # Stage 1 — frozen encoder, classifier head only.
    teacher.freeze_encoder()
    opt = torch.optim.AdamW([p for p in teacher.parameters() if p.requires_grad],
                            lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TEACHER_STAGE1_EPOCHS, eta_min=1e-6)
    for epoch in range(1, TEACHER_STAGE1_EPOCHS + 1):
        run_classifier_epoch(teacher, train_loader, opt, True,
                             f"teacher stage1 {epoch}/{TEACHER_STAGE1_EPOCHS} train")
        _, val_top1 = run_classifier_epoch(teacher, val_loader, opt, False,
                                           f"teacher stage1 {epoch}/{TEACHER_STAGE1_EPOCHS} val")
        maybe_save(val_top1)
        sched.step()
        print(f"teacher stage1 {epoch}/{TEACHER_STAGE1_EPOCHS} val_top1={val_top1:.2f}")

    # Stage 2 — unfreeze last block, fine-tune with a smaller LR.
    teacher.unfreeze_last_block()
    opt = torch.optim.AdamW([p for p in teacher.parameters() if p.requires_grad],
                            lr=TEACHER_STAGE2_LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=TEACHER_STAGE2_EPOCHS, eta_min=1e-7)
    for epoch in range(1, TEACHER_STAGE2_EPOCHS + 1):
        run_classifier_epoch(teacher, train_loader, opt, True,
                             f"teacher stage2 {epoch}/{TEACHER_STAGE2_EPOCHS} train")
        _, val_top1 = run_classifier_epoch(teacher, val_loader, opt, False,
                                           f"teacher stage2 {epoch}/{TEACHER_STAGE2_EPOCHS} val")
        maybe_save(val_top1)
        sched.step()
        print(f"teacher stage2 {epoch}/{TEACHER_STAGE2_EPOCHS} val_top1={val_top1:.2f}")

    teacher.load_state_dict(best_state)
    print(f"best teacher val_top1={best_top1:.2f}")
    return teacher


teacher = adapt_teacher()
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

## 3. The best student — built from scratch

The student is a lightweight convolutional **encoder** followed by a **convolutional predictor** that maps its channels into the teacher's pre-GAP feature-map channels. 

Each encoder block is `Conv3x3 → BatchNorm → SiLU → MaxPool`, halving the spatial resolution, so five blocks take a 224×224 image down to a 7×7 map — matching the teacher's feature-map resolution.

In [ ]:
def conv_block(in_ch: int, out_ch: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.SiLU(inplace=True),
        nn.MaxPool2d(kernel_size=2, stride=2),
    )


class StudentEncoder(nn.Module):
    def __init__(self, channels: list[int]) -> None:
        super().__init__()
        blocks, in_ch = [], 3
        for out_ch in channels:
            blocks.append(conv_block(in_ch, out_ch))
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.feature_channels = channels[-1]

    def forward(self, x: Tensor) -> Tensor:
        return self.features(x)  # pre-GAP feature map


class ConvFeaturePredictor(nn.Module):
    """1x1 convolutions mapping student channels into the teacher feature-map channels."""

    def __init__(self, in_ch: int, out_ch: int) -> None:
        super().__init__()
        hidden = max(256, min(1024, out_ch // 2))
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, hidden, kernel_size=1, bias=False),
            nn.BatchNorm2d(hidden),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden, out_ch, kernel_size=1),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.net(x)


class StudentDistillationModel(nn.Module):
    """Student-L encoder + convolutional predictor -> teacher-compatible feature map."""

    def __init__(self, channels: list[int], teacher_channels: int) -> None:
        super().__init__()
        self.encoder = StudentEncoder(channels)
        self.predictor = ConvFeaturePredictor(self.encoder.feature_channels, teacher_channels)

    def forward(self, x: Tensor) -> Tensor:
        return self.predictor(self.encoder(x))


student = StudentDistillationModel(STUDENT_CHANNELS, teacher.feature_channels).to(DEVICE)
print(student)

## 4. Distillation (MSE + CrossEntropy)

For each image the frozen teacher yields its pre-GAP feature map.  
The student predicts that map; the MSE term matches it directly, while the CE term routes the student's prediction through the frozen teacher pooling + classifier and supervises it with the true label:

$$L = \alpha \cdot \text{MSE}(\hat{f}_s, f_t) + \beta \cdot \text{CE}(\text{classifier}(\text{GAP}(\hat{f}_s)), y)$$

In [ ]:
def align_spatial(student_map: Tensor, teacher_map: Tensor) -> Tensor:
    if student_map.shape[-2:] != teacher_map.shape[-2:]:
        return F.adaptive_avg_pool2d(student_map, teacher_map.shape[-2:])
    return student_map


def run_distill_epoch(student, optimizer, loader, train: bool, desc: str):
    student.train(train)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    totals = {"mse": 0.0, "top1": 0.0, "top5": 0.0, "n": 0}
    progress = tqdm(loader, desc=desc, leave=False, unit="batch")
    for images, labels in progress:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        with torch.no_grad():
            teacher_map, _ = teacher.forward_features(images)
        with torch.set_grad_enabled(train), torch.amp.autocast(DEVICE.type, enabled=AMP):
            student_map = student(images)
            logits = teacher.classify_feature_map(student_map)
            mse = F.mse_loss(align_spatial(student_map, teacher_map), teacher_map)
            loss = ALPHA * mse + BETA * F.cross_entropy(logits, labels)
        if train:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        bs = labels.size(0)
        top1, top5 = accuracy(logits, labels)
        totals["mse"] += mse.item() * bs
        totals["top1"] += top1 * bs
        totals["top5"] += top5 * bs
        totals["n"] += bs
        progress.set_postfix(mse=f"{totals['mse'] / totals['n']:.3f}",
                             top1=f"{totals['top1'] / totals['n']:.2f}")
    n = totals["n"]
    return totals["mse"] / n, totals["top1"] / n, totals["top5"] / n


optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=DISTILL_EPOCHS, eta_min=1e-6)
best_state, best_top1 = None, -1.0
for epoch in range(1, DISTILL_EPOCHS + 1):
    run_distill_epoch(student, optimizer, train_loader, train=True,
                      desc=f"distill {epoch}/{DISTILL_EPOCHS} train")
    val_mse, val_top1, val_top5 = run_distill_epoch(student, optimizer, val_loader, train=False,
                                                     desc=f"distill {epoch}/{DISTILL_EPOCHS} val")
    if val_top1 > best_top1:
        best_top1 = val_top1
        best_state = {k: v.detach().cpu().clone() for k, v in student.state_dict().items()}
    scheduler.step()
    print(f"distill {epoch}/{DISTILL_EPOCHS} val_top1={val_top1:.2f} val_top5={val_top5:.2f} val_mse={val_mse:.4f}")

student.load_state_dict(best_state)
print(f"best student val_top1={best_top1:.2f}")

## 5. Evaluation, cost savings, and saved outputs

Report the student's test Top-1/Top-5 and its parameter / GFLOPs savings against the teacher.  
The cost is measured on the deployed path: student encoder + predictor + frozen teacher pooling and classifier.

The student weights and a metrics summary are written to `notebooks/outputs/`.

In [ ]:
class DeployedStudent(nn.Module):
    """Student feature map -> frozen teacher head (pooling + classifier)."""

    def __init__(self, student, teacher):
        super().__init__()
        self.student, self.pool, self.classifier = student, teacher.pool, teacher.classifier

    def forward(self, x: Tensor) -> Tensor:
        return self.classifier(torch.flatten(self.pool(self.student(x)), 1))


@torch.no_grad()
def evaluate(model, loader, desc: str):
    model.eval()
    totals = {"top1": 0.0, "top5": 0.0, "n": 0}
    progress = tqdm(loader, desc=desc, leave=False, unit="batch")
    for images, labels in progress:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        top1, top5 = accuracy(model(images), labels)
        bs = labels.size(0)
        totals["top1"] += top1 * bs
        totals["top5"] += top5 * bs
        totals["n"] += bs
        progress.set_postfix(top1=f"{totals['top1'] / totals['n']:.2f}")
    return totals["top1"] / totals["n"], totals["top5"] / totals["n"]


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def gflops(model):
    try:
        from thop import profile
        example = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
        macs, _ = profile(model, inputs=(example,), verbose=False)
        return macs * 2 / 1e9
    except Exception:
        return None


deployed = DeployedStudent(student, teacher).to(DEVICE)
s_top1, s_top5 = evaluate(deployed, test_loader, "test (student)")
t_top1, t_top5 = evaluate(teacher, test_loader, "test (teacher)")
s_params, t_params = count_params(deployed), count_params(teacher)
s_gflops, t_gflops = gflops(deployed), gflops(teacher)
params_saved_pct = 100 * (1 - s_params / t_params)
gflops_saved_pct = 100 * (1 - s_gflops / t_gflops) if s_gflops and t_gflops else None

print(f"Best student ({DATASET} / {TEACHER}): Student-L + pre-GAP + MSE+CE")
print(f"  student  top1={s_top1:.2f}%  top5={s_top5:.2f}%  params={s_params/1e6:.2f}M  gflops={s_gflops}")
print(f"  teacher  top1={t_top1:.2f}%  top5={t_top5:.2f}%  params={t_params/1e6:.2f}M  gflops={t_gflops}")
print(f"  saved    params={params_saved_pct:.1f}%"
      + (f"  gflops={gflops_saved_pct:.1f}%" if gflops_saved_pct is not None else ""))

run_name = f"{DATASET}_{TEACHER}_student_l_pregap_mse_ce"
torch.save(student.state_dict(), OUTPUT_DIR / f"{run_name}.pt")
summary = {
    "dataset": DATASET, "teacher": TEACHER, "student": "student_l",
    "target": TARGET, "loss": "mse_ce",
    "student_top1": s_top1, "student_top5": s_top5,
    "teacher_top1": t_top1, "teacher_top5": t_top5,
    "student_params": s_params, "teacher_params": t_params,
    "student_gflops": s_gflops, "teacher_gflops": t_gflops,
    "params_saved_pct": params_saved_pct, "gflops_saved_pct": gflops_saved_pct,
}
(OUTPUT_DIR / f"{run_name}_summary.json").write_text(json.dumps(summary, indent=2))
print(f"Saved {OUTPUT_DIR / (run_name + '.pt')} and {OUTPUT_DIR / (run_name + '_summary.json')}")